In [ ]:
import torch
from PIL import Image, ImageOps
from transformers import DetrImageProcessor, DetrForObjectDetection
import numpy as np 

def image_detection(frame) :
    image = Image.open(frame)
    image = ImageOps.exif_transpose(image)
    image = image.convert("RGB")

    processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50", revision="no_timm")
    model = DetrForObjectDetection.from_pretrained("isalia99/detr-resnet-50-sku110k")
    model.eval()

    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)


    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.7)[0]
    for box in results["boxes"]:
         box_np = box.cpu().numpy().astype(int)  
         bbox.append(box_np)
         
 return bbox 


In [ ]:
import cv2 
import pandas as pd 
import time 
import csv 
import mediapipe as mp 
from ultralytics import YOLO 


def reset_csv():
    with open('data.csv' , 'w') as new_file   
        feild  = ['bbox1' , 'bbox2' , 'bbox3' , 'bbox4' , 'item_id'] 
        csv_writer = csv.DictWriter(new_file , feild names  = feild)
        csv_writer.writeheader()

with open('face.csv' , 'w') as new_file   
        feild  = ['Detection'  , 'person_id'] 
        csv_writer = csv.DictWriter(new_file , feild names  = feild)
        csv_writer.writeheader()

person = 1 
def face_landmarks(frame): 
     face = mp.solutions.face_mesh
     a = []  
     b = []  
     c = []
     result = [] 
     with face.FaceMesh(max_num_faces = 1  , 
                     refine_landmarks = True ,
                     min_detection_confidence = 0.5 , 
                     min_tracking_confidence = 0.5 ) as mesh :
          bgr = cv2.cvtColor(frame , cv2.COLOR_BGR2RGB)
          out =  mesh.process(bgr) 
          if out.multi_face_landmarks:
             for face_landmarks in out.multi_face_landmarks :
               for landmark in face_landmarks.landmark :
                 a.append(landmark.x)
                 b.append(landmark.y)
                 c.append(landmark.z) 
             for j in range(len(a)) :
               result.append(a[j] - min(a)) 
               result.append(b[j] - min(b)) 
               result.append(c[j] - min(c)) 
     return result 


def track_person(frame):
     det = face_landmarks(frame) 
     with open('face.csv' , 'a') as file:
          csv_writer = csv.writer(file)
          for detections  in det : 
              csv_writer.writerow([detections, person ])
              person = person +1 

    
  
model = YOLO("yolov8n.pt")


with open('data.csv' , 'w') as new_file:
 feild  = ['bbox1' , 'bbox2' , 'bbox3' , 'bbox4' , 'id'] 
 csv_writer = csv.DictWriter(new_file , feild names  = feild)
 csv_writer.writeheader()


human_face = 0 
counter = 1 
webcam  = cv2.VideoCapture(0) 
last_time = -1 

while True: 
    ret , frame = webcam.read() 
    df = pd.read_csv('data.csv')

    if ret :
          if df.empty :
            bbox = image_detection(frame)
            with open('data.csv' , 'a') as file : 
             csv_writer = csv.writer(file)
             for box in bbox : 
              csv_writer.writerow([box[0] , box[1] , box[2] , box[3] , counter ]) 
              counter = counter +1 
              last_time  = time.time() 

          results = model(frame)
          for box in results[0].boxes: 
              if box.cls[0].item is 0  :
                face = pd.read_csv(face.csv) 
                check = face_landmarks(frame) 
                if face['check'].isna():  
                 for box in results[0].boxes: 
                  if box.cls[0].item == 0 :
                    track_person(frame)
                  break 



        if last_time is not -1 and time.time()-last_time is 600  : 
           results = model(frame)
           for box in results[0].boxes: 
              if box.cls[0].item is 0  :
                 human_face = 1 
                 break

           while human_face is 1 : 
              time.sleep(1)
              results = model(frame)
              human_face =  0 
              for box in results[0].boxes: 
                  if box.cls[0].item == 0 :
                    human_face = 1 
                    break
                  
           reset_csv() 
           with open('data.csv' , 'a') as file :
             csv_writer = csv.writer(file)
             for box in bbox : 
                 csv_writer.writerow([box[0] , box[1] , box[2] , box[3] , counter ]) 
                 counter = counter +1 
                 last_time  = time.time()
                 
webcam.release()
cv2.destroyAllWindows() 
          
           
           
           






